# Import libraries and load the CRM dataset
This loads the dataset from Stage 3, including the CRM segments.

In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv("../data/processed/gamepulse_players_with_crm_segments.csv")

df.head()

,player_id,install_date,country,device_type,acquisition_channel,campaign_name,sessions_day_1,sessions_day_7,total_playtime_minutes,levels_completed,in_app_purchases_gbp,ad_views,retained_day_1,retained_day_7,retained_day_30,churned,basic_player_segment,crm_segment
0,P00001,2026-02-21,India,Android,Facebook Ads,FB_Retargeting,6,8,172.9,12,0.0,13,1,1,0,1,Churned,Churned Player
1,P00002,2026-01-15,United States,Android,Google Ads,GAds_Display,2,2,88.9,5,0.0,5,1,1,0,1,Churned,Churned Player
2,P00003,2026-03-13,India,iOS,Google Ads,GAds_AppInstall,5,3,103.8,10,0.0,6,1,1,0,1,Churned,Churned Player
3,P00004,2026-03-02,Japan,iOS,Google Ads,GAds_Display,4,4,109.8,10,0.0,5,1,1,0,1,Churned,Churned Player
4,P00005,2026-01-21,Canada,Android,Google Ads,GAds_AppInstall,1,0,39.6,6,0.0,0,1,0,0,1,At-Risk Early Drop-off,At-Risk Early Drop-off Player


# Create a SQLite database
This creates a small database called:
    gamepulse.db
and stores your dataset inside a table called:
    players

In [2]:
conn = sqlite3.connect("../data/processed/gamepulse.db")

df.to_sql("players", conn, if_exists="replace", index=False)

print("Gaming player data loaded into SQLite database successfully.")

Gaming player data loaded into SQLite database successfully.


# Check the table
This is like df.head(), but using SQL.

In [3]:
query = """
SELECT *
FROM players
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,player_id,install_date,country,device_type,acquisition_channel,campaign_name,sessions_day_1,sessions_day_7,total_playtime_minutes,levels_completed,in_app_purchases_gbp,ad_views,retained_day_1,retained_day_7,retained_day_30,churned,basic_player_segment,crm_segment
0,P00001,2026-02-21,India,Android,Facebook Ads,FB_Retargeting,6,8,172.9,12,0.0,13,1,1,0,1,Churned,Churned Player
1,P00002,2026-01-15,United States,Android,Google Ads,GAds_Display,2,2,88.9,5,0.0,5,1,1,0,1,Churned,Churned Player
2,P00003,2026-03-13,India,iOS,Google Ads,GAds_AppInstall,5,3,103.8,10,0.0,6,1,1,0,1,Churned,Churned Player
3,P00004,2026-03-02,Japan,iOS,Google Ads,GAds_Display,4,4,109.8,10,0.0,5,1,1,0,1,Churned,Churned Player
4,P00005,2026-01-21,Canada,Android,Google Ads,GAds_AppInstall,1,0,39.6,6,0.0,0,1,0,0,1,At-Risk Early Drop-off,At-Risk Early Drop-off Player


# Total players and total revenue
This answers:

How many players are in the dataset and how much revenue was generated?

In [4]:
query = """
SELECT 
    COUNT(DISTINCT player_id) AS total_players,
    ROUND(SUM(in_app_purchases_gbp), 2) AS total_revenue
FROM players;
"""

pd.read_sql_query(query, conn)

,total_players,total_revenue
0,1200,1367.35


# Overall retention and churn
This gives the same retention KPIs as Stage 2, but now using SQL.

In [5]:
query = """
SELECT
    ROUND(AVG(retained_day_1) * 100, 1) AS day_1_retention,
    ROUND(AVG(retained_day_7) * 100, 1) AS day_7_retention,
    ROUND(AVG(retained_day_30) * 100, 1) AS day_30_retention,
    ROUND(AVG(churned) * 100, 1) AS churn_rate
FROM players;
"""

pd.read_sql_query(query, conn)

,day_1_retention,day_7_retention,day_30_retention,churn_rate
0,94.8,84.0,40.9,59.1


# Retention by acquisition channel
This shows which acquisition channels brought better long-term players.

In [6]:
query = """
SELECT
    acquisition_channel,
    COUNT(DISTINCT player_id) AS players,
    ROUND(AVG(retained_day_1) * 100, 1) AS day_1_retention,
    ROUND(AVG(retained_day_7) * 100, 1) AS day_7_retention,
    ROUND(AVG(retained_day_30) * 100, 1) AS day_30_retention,
    ROUND(AVG(churned) * 100, 1) AS churn_rate,
    ROUND(SUM(in_app_purchases_gbp), 2) AS revenue
FROM players
GROUP BY acquisition_channel
ORDER BY day_30_retention DESC;
"""

retention_by_channel_sql = pd.read_sql_query(query, conn)

retention_by_channel_sql

,acquisition_channel,players,day_1_retention,day_7_retention,day_30_retention,churn_rate,revenue
0,Organic,339,95.9,86.7,46.9,53.1,491.48
1,App Store Search,111,95.5,88.3,45.0,55.0,101.85
2,Facebook Ads,167,97.0,84.4,39.5,60.5,165.82
3,Google Ads,243,93.8,86.4,38.3,61.7,345.60
4,Influencer Campaign,149,91.9,76.5,36.2,63.8,169.76
5,TikTok Ads,191,94.2,79.1,36.1,63.9,92.84


# Churn by country
churn_by_country_sql

This helps identify which countries may need stronger CRM or onboarding support.

In [7]:
query = """
SELECT
    country,
    COUNT(DISTINCT player_id) AS players,
    ROUND(AVG(retained_day_30) * 100, 1) AS day_30_retention,
    ROUND(AVG(churned) * 100, 1) AS churn_rate,
    ROUND(SUM(in_app_purchases_gbp), 2) AS revenue
FROM players
GROUP BY country
ORDER BY churn_rate DESC;
"""

churn_by_country_sql = pd.read_sql_query(query, conn)

churn_by_country_sql

,country,players,day_30_retention,churn_rate,revenue
0,Japan,74,32.4,67.6,37.93
1,United Kingdom,209,34.4,65.6,217.74
2,Brazil,88,36.4,63.6,111.82
3,South Korea,75,38.7,61.3,41.90
4,India,124,38.7,61.3,149.84
5,Canada,80,40.0,60.0,171.83
6,France,104,40.4,59.6,178.81
7,United States,260,45.4,54.6,241.72
8,Germany,99,50.5,49.5,150.89
9,Nigeria,87,50.6,49.4,64.87


# CRM segment performance
This shows which player segment is most valuable.

In [8]:
query = """
SELECT
    crm_segment,
    COUNT(DISTINCT player_id) AS players,
    ROUND(AVG(sessions_day_7), 1) AS avg_sessions_day_7,
    ROUND(AVG(total_playtime_minutes), 1) AS avg_playtime_minutes,
    ROUND(AVG(levels_completed), 1) AS avg_levels_completed,
    ROUND(SUM(in_app_purchases_gbp), 2) AS total_revenue,
    ROUND(AVG(retained_day_30) * 100, 1) AS day_30_retention,
    ROUND(AVG(churned) * 100, 1) AS churn_rate
FROM players
GROUP BY crm_segment
ORDER BY total_revenue DESC;
"""

crm_segment_sql = pd.read_sql_query(query, conn)

crm_segment_sql

,crm_segment,players,avg_sessions_day_7,avg_playtime_minutes,avg_levels_completed,total_revenue,day_30_retention,churn_rate
0,High-Value Retained Player,94,8.4,184.4,12.6,932.06,100.0,0.0
1,Churned Player,582,4.7,123.7,8.9,368.37,0.0,100.0
2,At-Risk Early Drop-off Player,154,0.5,56.7,5.0,66.92,17.5,82.5
3,New or Casual Player,112,2.9,97.2,7.7,0.00,100.0,0.0
4,Loyal Engaged Player,258,9.4,195.3,13.0,0.00,100.0,0.0


# Identify high-value retained players
This gives you the top high-value players that the business should protect and reward.

In [9]:
query = """
SELECT
    player_id,
    country,
    device_type,
    acquisition_channel,
    sessions_day_7,
    total_playtime_minutes,
    levels_completed,
    in_app_purchases_gbp,
    crm_segment
FROM players
WHERE crm_segment = 'High-Value Retained Player'
ORDER BY in_app_purchases_gbp DESC
LIMIT 20;
"""

high_value_players_sql = pd.read_sql_query(query, conn)

high_value_players_sql

,player_id,country,device_type,acquisition_channel,sessions_day_7,total_playtime_minutes,levels_completed,in_app_purchases_gbp,crm_segment
0,P00244,United States,iOS,Google Ads,13,250.5,17,49.99,High-Value Retained Player
1,P00367,Germany,iOS,Facebook Ads,4,101.9,9,49.99,High-Value Retained Player
2,P00417,United States,iOS,Organic,1,32.2,7,49.99,High-Value Retained Player
3,P00421,United Kingdom,Android,Organic,21,429.1,27,49.99,High-Value Retained Player
4,P00476,India,PC,Organic,5,92.5,9,49.99,High-Value Retained Player
5,P00508,Germany,PC,Google Ads,18,399.5,25,49.99,High-Value Retained Player
6,P00910,Brazil,Android,Influencer Campaign,5,145.6,10,49.99,High-Value Retained Player
7,P00223,Canada,Android,App Store Search,3,84.0,7,19.99,High-Value Retained Player
8,P00482,Canada,Android,Organic,8,135.5,11,19.99,High-Value Retained Player
9,P00509,France,iOS,Google Ads,10,254.0,17,19.99,High-Value Retained Player


# Identify at-risk players
This gives players who may need early CRM support before they fully churn.

In [10]:
query = """
SELECT
    player_id,
    country,
    device_type,
    acquisition_channel,
    sessions_day_1,
    sessions_day_7,
    total_playtime_minutes,
    levels_completed,
    crm_segment
FROM players
WHERE crm_segment = 'At-Risk Early Drop-off Player'
ORDER BY total_playtime_minutes DESC
LIMIT 20;
"""

at_risk_players_sql = pd.read_sql_query(query, conn)

at_risk_players_sql

,player_id,country,device_type,acquisition_channel,sessions_day_1,sessions_day_7,total_playtime_minutes,levels_completed,crm_segment
0,P00318,United Kingdom,iOS,TikTok Ads,5,1,89.8,7,At-Risk Early Drop-off Player
1,P00044,Japan,Android,Facebook Ads,2,0,89.6,5,At-Risk Early Drop-off Player
2,P00258,Nigeria,Android,TikTok Ads,1,0,89.5,9,At-Risk Early Drop-off Player
3,P00661,United States,Android,Organic,4,1,89.4,6,At-Risk Early Drop-off Player
4,P00291,United States,Android,TikTok Ads,3,0,89.2,8,At-Risk Early Drop-off Player
5,P00734,Canada,iOS,Google Ads,2,1,88.7,7,At-Risk Early Drop-off Player
6,P01164,United States,iOS,Influencer Campaign,3,1,87.7,9,At-Risk Early Drop-off Player
7,P00528,United States,iOS,TikTok Ads,5,0,87.6,7,At-Risk Early Drop-off Player
8,P00652,France,Android,Facebook Ads,2,1,87.1,7,At-Risk Early Drop-off Player
9,P00124,United States,Android,TikTok Ads,3,1,86.6,9,At-Risk Early Drop-off Player


# Revenue by device type
This helps you understand which device type brings better engagement or revenue.

In [11]:
query = """
SELECT
    device_type,
    COUNT(DISTINCT player_id) AS players,
    ROUND(SUM(in_app_purchases_gbp), 2) AS revenue,
    ROUND(AVG(retained_day_30) * 100, 1) AS day_30_retention,
    ROUND(AVG(churned) * 100, 1) AS churn_rate
FROM players
GROUP BY device_type
ORDER BY revenue DESC;
"""

revenue_by_device_sql = pd.read_sql_query(query, conn)

revenue_by_device_sql

,device_type,players,revenue,day_30_retention,churn_rate
0,iOS,540,630.26,39.8,60.2
1,Android,549,577.23,41.2,58.8
2,PC,111,159.86,45.0,55.0


# Save SQL analysis results


In [12]:
import os

os.makedirs("../reports", exist_ok=True)

retention_by_channel_sql.to_csv("../reports/stage4_sql_retention_by_channel.csv", index=False)
churn_by_country_sql.to_csv("../reports/stage4_sql_churn_by_country.csv", index=False)
crm_segment_sql.to_csv("../reports/stage4_sql_crm_segment_performance.csv", index=False)
high_value_players_sql.to_csv("../reports/stage4_sql_high_value_players.csv", index=False)
at_risk_players_sql.to_csv("../reports/stage4_sql_at_risk_players.csv", index=False)
revenue_by_device_sql.to_csv("../reports/stage4_sql_revenue_by_device.csv", index=False)

print("Stage 4 SQL analysis files saved successfully.")

Stage 4 SQL analysis files saved successfully.


# Close the database connection

In [13]:
conn.close()

print("Database connection closed.")

Database connection closed.


# ## Stage 4 Business Insight: SQL Analysis

The SQL analysis confirms the key findings from the Python analysis. The data shows that long-term retention varies across acquisition channels, countries and CRM player segments. Organic and App Store Search channels produced stronger long-term retention, while some paid or influencer-led channels showed higher churn. This suggests that user acquisition should be judged not only by install volume, but also by player quality and retention.

The CRM segment analysis shows that high-value retained players generated the strongest revenue despite being a smaller group. This means the business should protect this segment with loyalty rewards, exclusive offers and personalised engagement. At-risk early drop-off players should receive onboarding support and early re-engagement messages, while churned players may need win-back campaigns or feedback surveys.

Using SQL alongside Python strengthens the project because it shows that the same business questions can be answered through database queries, not only through notebook analysis. This is useful for data analyst, product analyst and CRM analytics roles where SQL is often used to extract and summarise business data.